# Fine-Tuning LLaMA 3.2 Vision for Medical Image Understanding

## Aim
To adapt a pre-trained multimodal vision-language model
(LLaMA 3.2 Vision) to generate medically relevant descriptions
of radiology images using parameter-efficient fine-tuning (LoRA).

This work is for **educational and research purposes only**.

In [1]:
# verify pyTorch sees the GPU

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")


True
Tesla T4
14.74127197265625 GB


In [1]:
!pip install pillow==11.3.0

In [2]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer


In [3]:
from huggingface_hub import login, logout
logout()
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Not logged in!


In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
# 🔑 Memory-saving settings (correct place)
model.config.use_cache = False
model.gradient_checkpointing_enable()

In [7]:
!pip install medmnist

In [8]:
from medmnist import PneumoniaMNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np

In [9]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = PneumoniaMNIST(
    split='train',
    transform=transform,
    download=True
)

In [10]:
import numpy as np
from PIL import Image

LABEL_MAP = {
    0: "a normal chest X-ray with no visible abnormalities",
    1: "a chest X-ray showing signs consistent with pneumonia"
}

def format_example(idx):
    img, label = train_dataset[idx]

    # Extract scalar label safely
    label = int(label[0]) if hasattr(label, "__len__") else int(label)

    # Tensor → PIL Image (KEEP as PIL, NOT bytes)
    img_pil = Image.fromarray(
        (img.squeeze().numpy() * 255).astype(np.uint8)
    )

    return {
        # 🔑 Actual image goes here
        "images": [img_pil],

        # 🔑 Messages contain ONLY placeholders
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "You are a medical imaging assistant. "
                            "Describe this chest X-ray using clinical terminology. "
                            "Do not provide diagnosis or treatment."
                        )
                    },
                    {
                        "type": "image"
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": LABEL_MAP[label]
                    }
                ]
            }
        ]
    }


In [11]:
from datasets import Dataset

train_data = [format_example(i) for i in range(150)]
hf_train_dataset = Dataset.from_list(train_data)

In [12]:
hf_train_dataset[0].keys()

dict_keys(['images', 'messages'])

In [13]:
# Apply LoRA (Parameter-efficient fine-tuning)

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                     # SAFE for free Colab
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,898,240 || all params: 10,676,119,075 || trainable%: 0.0552


In [14]:
from transformers import TrainingArguments
from trl import SFTTrainer

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/medical_llama_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # ⬅️ increase this
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)


In [16]:
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train_dataset,
    processing_class=processor,
    args=training_args
)

In [17]:
trainer.train()


# ---------- AUTO SAVE ----------
SAVE_DIR = "/content/medical_llama_lora"

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("✅ LoRA adapters saved.")

# ---------- AUTO ZIP ----------
import shutil
zip_path = "/content/medical_llama_lora.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", SAVE_DIR)

print("✅ Model zipped.")

# ---------- AUTO DOWNLOAD ----------
from google.colab import files
files.download(zip_path)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'bos_token_id': 128000, 'pad_token_id': 128004}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 126.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 136.12 MiB is free. Process 162995 has 14.61 GiB memory in use. Of the allocated memory 14.35 GiB is allocated by PyTorch, and 129.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## test the finetuned model

In [ ]:
sample = hf_train_dataset[0]["messages"]

inputs = processor.apply_chat_template(
    sample,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

print(processor.decode(outputs[0], skip_special_tokens=True))